In [ ]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler


DATA_PATH = "/content/gdrive/MyDrive/REU/PROJECT/DATA/"

from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
annotated_data = pd.read_csv(DATA_PATH+"annotated_data.csv")

In [ ]:
annotated_data

,Unnamed: 0,full_name,transactor_type
0,0,franklin county reagan coalition,Committee
1,1,nextgen climate action committee,Committee
2,2,zarwin baum good government pac,Committee
3,3,mid-atlantic laborers' political league,Committee
4,4,toll bros inc pac,Committee
...,...,...,...
518282,1048568,cindy olofson,Individual
518283,1048569,ryan olsen,Individual
518284,1048570,andrew orgill,Individual
518285,1048572,chris ortega,Individual


<h1>CONVERT TO BINARY LABELS</h1>

In [ ]:
def org_labels(st):
    if st != "Individual" and st != "Candidate":
        return "Organization"
    return "Individual"

In [ ]:
annotated_data['transactor_type'] = annotated_data['transactor_type'].apply(org_labels)
annotated_data = annotated_data[['full_name', 'transactor_type']].copy()

In [ ]:
annotated_data.dropna(subset='full_name', inplace=True)

In [ ]:
annotated_data['full_name'].sort_values()

,full_name
9580,citizens for boyle
8800,"mauck, shawn c"
360485,!jayne 2012
95841,"""""rip"""" stephen wilson"
323972,"""a company, inc. phx portable restooms"""
...,...
353475,zygmunt roguski
94328,zylphia cummins
193258,zymages
433906,zyra brown


<h1>Train, Test, split</h1>

In [ ]:

def convert_bool(input_label):
    '''Converts to numerical encoding'''
    if input_label == 'Candidate' or input_label == 'Individual':
        return 1
    else:
        return 0

# encode transactor_type as numeric labels
annotated_data['label'] = np.vectorize(convert_bool)(annotated_data['transactor_type'])

# features and target for under-sampling
X = annotated_data[['full_name']]
y = annotated_data['label']

# apply random under-sampling to balance the classes
rus = RandomUnderSampler(random_state = 42)
X_resampled, y_resampled = rus.fit_resample(X, y)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.20, random_state=24)


In [ ]:
X_train

,full_name
393376,gavin kowalski
223808,janet emel
473366,a small orange
228749,a pac for employees of safeway inc
49242,debbie ricke
...,...
443751,melinda t bishop-morfin
18613,sharon e huie-lew
7657,"newtown democrats, regina gairo, treasurer"
458214,driss ferza


In [ ]:
y_train

,label
393376,1
223808,1
473366,0
228749,0
49242,1
...,...
443751,1
18613,1
7657,0
458214,1


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.125, random_state=24)


In [ ]:
X_test

,full_name
507706,quail creek crossing
225326,lyn t ward
141944,the governors
404422,briteverify
258567,u.s. american
...,...
109541,national pen company
146726,colleen r floyd
475604,robin cerillo
482662,kenton rael


In [ ]:
y_test

,label
507706,0
225326,1
141944,0
404422,0
258567,0
...,...
109541,0
146726,1
475604,1
482662,1


In [ ]:
train_set = X_train.copy()
train_set['label'] = y_train
train_set

,full_name,label
415060,shell station,0
84690,mary brandenburg campaign,0
470782,ann perez,1
5636,"nelson, eric friends of",0
411413,rim resort gas,0
...,...,...
50007,terri rowe,1
277927,lilo's coffee,0
238430,"baker, sally",0
78926,youth on their own,0


In [ ]:
validation_set = X_val.copy()
validation_set['label'] = y_val
validation_set

,full_name,label
21375,plumbers & steamfitters local,0
211228,leona bennett,1
261969,"manross 2008, scottsdale for",0
268117,the symington group llc,0
292382,betsy buford,1
...,...,...
417247,mara aspinall,1
427554,lorraine m ryall,1
409689,manuel galvan,1
109126,mc frugals,0


In [ ]:
test_set = X_test.copy()
test_set['label'] = y_test
test_set

,full_name,label
507706,quail creek crossing,0
225326,lyn t ward,1
141944,the governors,0
404422,briteverify,0
258567,u.s. american,0
...,...,...
109541,national pen company,0
146726,colleen r floyd,1
475604,robin cerillo,1
482662,kenton rael,1


In [ ]:
train_set.to_csv("train.csv")
validation_set.to_csv("validation.csv")
test_set.to_csv("test.csv")